<center><h1>Soundalgekar_Abhishek_Project_Resnet101</h1></center>

Name: Abhishek Soundalgekar
<br>
Github Username: ABHISHEK SOUNDALGEKAR
<br>
USC ID: 2089011000

In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet101
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.metrics import classification_report
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from typing import Dict
import tensorflow as tf
from tensorflow.keras.applications import ResNet101
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    BatchNormalization,
    Activation,
    Dropout
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam as AdamOpt
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [4]:
DATA_DIR = Path("../data/WasteSplit")
IMG_DIMS = (224, 224)
BATCH_SZ = 5
EPOCHS   = 100

augmented_preprocessor = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet.preprocess_input,
    rotation_range=15,             # small random rotations
    width_shift_range=0.15,        # horizontal shifts
    height_shift_range=0.15,       # vertical shifts
    zoom_range=0.15,               # random zoom
    shear_range=0.1,               # slight shearing
    brightness_range=(0.9, 1.1),   # slight brightness variation
    horizontal_flip=True           # random flips
)

def make_generator(
    subset_name: str,
    shuffle_data: bool = False
) -> tf.keras.preprocessing.image.DirectoryIterator:
    """
    Creates a DirectoryIterator for the given subset ('train', 'val', or 'test').
    """
    return augmented_preprocessor.flow_from_directory(
        directory=str(DATA_DIR / subset_name),
        target_size=IMG_DIMS,
        batch_size=BATCH_SZ,
        class_mode="categorical",
        shuffle=shuffle_data
    )

generators: Dict[str, tf.keras.preprocessing.image.DirectoryIterator] = {
    split: make_generator(split, shuffle_data=(split == "train"))
    for split in ("train", "val", "test")
}

train_loader = generators["train"]
val_loader   = generators["val"]
test_loader  = generators["test"]


Found 3323 images belonging to 9 classes.
Found 710 images belonging to 9 classes.
Found 719 images belonging to 9 classes.


In [6]:
feature_extractor = ResNet101(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3),
    name="resnet101_backbone"
)
feature_extractor.trainable = False

#L2 regularizer & block helper
l2_reg = l2(1e-2)

def dense_bn_act(x, units, tag):
    x = Dense(units, kernel_regularizer=l2_reg, name=f"{tag}_dense")(x)
    x = BatchNormalization(name=f"{tag}_bn")(x)
    return Activation("relu", name=f"{tag}_act")(x)

y = GlobalAveragePooling2D(name="gap")(feature_extractor.output)
y = dense_bn_act(y, 256, "headA")
y = dense_bn_act(y, 128, "headB")
y = Dropout(0.3, name="head_dropout")(y)
y = dense_bn_act(y, 64,  "headC")

outputs = Dense(
    train_loader.num_classes,
    activation="softmax",
    name="predictions"
)(y)

classifier_net = Model(
    inputs=feature_extractor.input,
    outputs=outputs,
    name="waste_resnet101_classifier"
)

#Adam & metrics
adam = AdamOpt(learning_rate=1e-4)
classifier_net.compile(
    optimizer=adam,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

#early stop + LR reduction
early_stop_cb = EarlyStopping(
    monitor="val_loss",
    patience=40,
    restore_best_weights=True,
    verbose=1
)
reduce_lr_cb = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=10,
    verbose=1
)

training_history = classifier_net.fit(
    train_loader,
    validation_data=val_loader,
    epochs=EPOCHS,
    callbacks=[early_stop_cb, reduce_lr_cb]
)


171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


/opt/anaconda3/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/100
 30/665 ━━━━━━━━━━━━━━━━━━━━ 5:39 534ms/step - accuracy: 0.0959 - loss: 9.7018

KeyboardInterrupt: 